# SCRIPT DESCRIPTION

Filter full gene matrix and subset brain region of interest. By subsetting a specific brain region, we enrich can improve cell typing and enrich for cell populations of interest (e.g dopaminergic neurons in Substantia Nigra), while reducing confounding signal from unrelated brain regions.

In this example, Substantia Nigra cells are filtered for downstream analysis.
Using the HVG data (top 3k genes), the SN cells are filtered and these are used to then filter the full dataset (unfiltered for QC).
A new AnnData file is saved locally for the filtered cells.

** code has been adapted from ASAP CRN Tutorials

# 1. Package Imports and Configuration

In [ ]:
# Core scientific computing and visualization libraries
import numpy as np
import pandas as pd
import scanpy as sc

# Standard library imports
import sys
import subprocess
import importlib
import warnings
import os
import gc
from pathlib import Path

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime

# 1. Data structure

Define paths and input files

In [ ]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"
local_resources = WS_FILES / "local_resources"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)
print("Local CRN resources directory: ", local_resources)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

CRN input data (e.g. HVG and full matrix AnnData) should be available in ~/workspace/ws_files/local_resources. If data does not exist, follow script 01.

adata ("asap-cohort.final.metadataset-integrated.h5ad") is the HVG files with dataset metadata included. It is ready to use for filtering cell types or brain regions of interest and subsetting the full dataset

In [ ]:
# set paths for raw data files
adata_local_path = local_resources / f"asap-cohort.final.metadataset-integrated.h5ad"
dataset_metadata_local_path = local_resources / f"asap-cohort-dataset-metadata.csv"

# load adata and metadata files ("raw data")
adata = sc.read_h5ad(adata_local_path, backed="r")
df = pd.read_csv(dataset_metadata_local_path, low_memory=False)

# 2. Data Assessment and Filtering 



### 2.1 Anatomical Coverage and Substantia Nigra Specificity

We first evaluate the anatomical distribution of samples using standardized brain region mappings.  

Both detailed and simplified brain region annotations are used to assess whether datasets include sufficient **Substantia Nigra** representation for this case study.


In [ ]:
df[["ASAP_team_id","region_level_2_name"]].value_counts()

In [ ]:
# define the region to filter (choose from list of region_level_2)
region = "Substantia nigra"
region_savename = region.lower().replace(" ", "_").replace("/", "_")

In [ ]:
# Define a local path for workshop files
region_data_path = WS_FILES / region_savename

# map my cells directories
region_input_dir = ( region_data_path / "input")
region_output_dir = ( region_data_path / "output")

# Make sure the directories exists
os.makedirs(region_input_dir, exist_ok=True)
os.makedirs(region_output_dir, exist_ok=True)

# Create the directory if it doesn't already exist
if not region_data_path.exists():
    region_data_path.mkdir(parents=True)

print(f"Local data directory ready at: {region_data_path}")

In [ ]:
print(df[df["region_level_2_name"] == region][["ASAP_team_id", "ASAP_dataset_id"]].value_counts())

#validating sample ID is unique
df.duplicated(subset="ASAP_sample_id").sum()
print(len(df[df["region_level_2_name"]== region]), "{region} samples out of" ,len(df), "total samples")

df[df["region_level_2_name"] == region][["ASAP_team_id", "biobank_name"]].value_counts()

# Check missingness and most common values
print("n_obs:", adata.n_obs)
print("missing region_level_2:", adata.obs["region_level_2"].isna().sum())

adata.obs["region_level_2"].value_counts(dropna=False).head(25)

In [ ]:
# identify substantia nigra cells
region_cells = adata.obs["region_level_2"] == region
# Final boolean mask for subsetting
include = region_cells
print(region_cells.sum())

### 2.2 Subsetting to Substantia Nigra Cells



In [ ]:
# create region subset
region_ad = adata[include].to_memory()
adata.file.close()  # close the original adata file

In [ ]:
region_ad.obs[["team","dataset", "biobank_name", "condition_id", "region_level_2"]].value_counts()

In [ ]:
region_ad.obs[["class_name"]].value_counts()

In [ ]:
# We will save this intermediate anndata which will contain:
# - Cells restricted to the Substantia Nigra (SN)
# - Highly variable genes (HVGs) only
# - Normalized and log-transformed expression values
# - PCA and other preprocessing results stored in .obsm

region_hvg_path = region_input_dir / f"asap-cohort.{region_savename}_hvg.h5ad"
region_ad.write_h5ad(region_hvg_path)

# remove files from memory
del region_ad
gc.collect()

In [ ]:
# reload hvg adata in backed read-only
region_hvg_path = region_input_dir / f"asap-cohort.{region_savename}_hvg.h5ad"
region_ad = sc.read_h5ad(region_hvg_path, backed="r")

### 2.3 Integrating with Full Gene Expression Matrix
The previous steps have been using a **highly variable gene (HVG)–restricted** AnnData object (top 3k genes). Downstream analysis such as cell typing and pseudobulk require the full gene expression matrix.

In this step, we load the full, unfiltered AnnData object and extract the values for the previously selected Substantia Nigra cells. These values are combined with curated metadata and embeddings from the HVG-based object to construct a new AnnData object that preserves full gene coverage while maintaining consistent cell annotations.

In [ ]:
# Define paths to the full, unfiltered AnnData object
full_adata_filename = local_resources / f"asap-cohort.merged_cleaned_unfiltered.h5ad"

In [ ]:
# Load the full gene expression matrix in backed mode
# (avoids loading the entire object into memory)var_ = full_adata.var.copy()

full_adata = sc.read_h5ad(full_adata_filename, backed="r")

In [ ]:
# Preserve full gene annotations
var_ = full_adata.var.copy()

# Extract expression values for Substantia Nigra cells only
# using cell identifiers from the HVG-based AnnData object
X = full_adata[region_ad.obs_names].X.copy()

# Close the backing file explicitly
full_adata.file.close()

In [ ]:
# Construct a new AnnData object with:
# - full gene expression values
# - SN-restricted cells
# - curated metadata and embeddings from the HVG-based object

region_full_ad = sc.AnnData(
    X=X,
    obs=region_ad.obs,
    var=var_,
    uns=region_ad.uns,
    obsm=region_ad.obsm,
)
region_full_ad

# 3. Data Export

The exported AnnData object serves as input for downstream analysis (e.g. preprocessing, integration, cell annotation)

This AnnData object contains:
- Cells restricted to the **Substantia Nigra (SN)**
- The **full gene expression matrix** for SN-derived cells
- Curated metadata and annotations stored in `.obs`
- Embeddings and preprocessing results (e.g., PCA, UMAP) carried forward from HVG-based analyses and stored in `.obsm`

> Notes
> - An HVG-restricted AnnData object is exported earlier for integration and clustering workflows.
> - The object exported here restores full gene coverage for downstream biological interpretation.



In [ ]:
# To avoid overwriting or losing previously computed results,
# we archive selected embeddings by renaming them with a leading underscore.
# This allows us to retain historical results while freeing the original keys
# for recomputation or alternative methods downstream.


for key in ['X_pca',
            'X_pca_harmony',
            'X_scANVI',
            'X_scVI',
            'X_umap']:
    archived_key = f"_{key}"
    # rename
    region_full_ad.obsm[archived_key] = region_full_ad.obsm.pop(key)
    

# Similarly, we archive selected cell-level annotations in `.obs`
# to preserve earlier labels, probabilities, and clustering results.
# Archived fields are prefixed with an underscore to indicate
# they reflect a previous analysis state.

old_obs_fields = [ 'cell_type',
 'phenotype',
 'rho',
 'prob',
 'class_name',
 'subclass_name',
 'supertype_name',
 '_scvi_batch',
 '_scvi_labels',
 'C_scANVI',
 'leiden_res_0.05',
 'leiden_res_0.10',
 'leiden_res_0.20',
 'leiden_res_0.40'
 ]

for key in old_obs_fields:
    archived_key = f"_{key}"
    # rename
    region_full_ad.obs[archived_key] = region_full_ad.obs.pop(key)

In [ ]:
# Save the curated AnnData object
region_full_samples_filename = region_input_dir / f"asap-cohort.{region_savename}_cells_full_genes_curated.h5ad"
region_full_ad.write_h5ad(region_full_samples_filename)